# 2. Splits: four regimes, and why leakage is assertedA split is a pure function of (pairs, regime, seed), written to disk with a fingerprint.Nothing else in the project partitions data.| | Held out | Question ||---|---|---|| S1 | random pairs | fill a gap for a known drug and protein || S2 | whole drugs | screen a new compound || S3 | whole targets | screen against a new protein || S4 | both | neither side ever seen |

In [ ]:
import sys, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")import numpy as np, pandas as pdpd.set_option("display.width", 180); pd.set_option("display.max_columns", 50)from dta import data as D, splits as Sfrom dta.config import DATASETS, SPLIT_SEEDSproc = {ds: D.load_processed(ds) for ds in DATASETS}

In [ ]:
rep = pd.read_csv("../data/splits/report_davis.csv")cols = ["setting","seed","n_train","n_val","n_test","n_unused","fingerprint"]display(rep[rep.seed == rep.seed.min()][[c for c in cols if c in rep.columns]])

## The assertionsEach regime claims a property, and `check_split` raises if it is violated. These areerrors rather than warnings because the failure they catch is invisible in the results:leakage makes numbers look *better*, so nothing downstream would complain.

In [ ]:
pairs = proc["davis"]["pairs"]; nl, nt = proc["davis"]["n_lig"], proc["davis"]["n_tgt"]a = S.load_split("davis", "S3_new_target", 2026)train_t = set(pairs.loc[a=="train","tgt_idx"]); test_t = set(pairs.loc[a=="test","tgt_idx"])print("targets shared between train and test:", len(train_t & test_t))bad = a.copy(); victim = pairs.index[(a=="test")][0]; bad.loc[victim] = "train"try:    S.check_split(pairs, bad, "S3_new_target")except AssertionError as e:    print("guard fired as intended:", e)

## S4 discards half-new pairs on purposeA pair that is new on only one side belongs to S2 or S3. Putting it in training wouldmake the S4 test entities warm; putting it in test would answer a different question.

In [ ]:
a4 = S.load_split("davis", "S4_both_new", 2026)print(a4.value_counts().to_dict())